## **GIN Model**

In [2]:


import time, random
import numpy as np
import torch
import torch.nn.functional as F
from tqdm import tqdm
from copy import deepcopy
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, log_loss
from torch_geometric.datasets import TUDataset
from torch_geometric.loader import DataLoader
from torch_geometric.nn import GINConv, global_mean_pool, global_max_pool
from torch.nn import BatchNorm1d

import phe as paillier
import tenseal as ts
import pandas as pd

# ---------------- USER KNOBS (reduced / demo-friendly) ----------------
RND = 42
N_FOLDS = 3              # fewer folds -> faster
GIN_HIDDEN = 64          # smaller model
GIN_EMB = 128
GIN_EPOCHS = 20          # fewer epochs
GIN_PATIENCE = 6
PAILLIER_SCALE = 5000
PAILLIER_KEY = 1024      # smaller key for demo speed (not for production)
CKKS_CTX_SCALE_EXP = 40
CKKS_BATCH_SIZE = 4
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DL_BATCH_SIZE = 16       # smaller DataLoader batches
# ---------------------------------------------------------------------

random.seed(RND); np.random.seed(RND); torch.manual_seed(RND)
device = torch.device(DEVICE)
print("Device:", device)

# ---------- GIN model used for embeddings ----------
class GINModel(torch.nn.Module):
    def __init__(self, in_ch, hidden=GIN_HIDDEN, emb_dim=GIN_EMB, dropout=0.3):
        super().__init__()
        def mlp(inp, outp):
            return torch.nn.Sequential(torch.nn.Linear(inp, outp), torch.nn.ReLU(), torch.nn.Linear(outp, outp))
        self.conv1 = GINConv(mlp(in_ch, hidden)); self.bn1 = BatchNorm1d(hidden)
        self.conv2 = GINConv(mlp(hidden, hidden));  self.bn2 = BatchNorm1d(hidden)
        self.conv3 = GINConv(mlp(hidden, hidden));  self.bn3 = BatchNorm1d(hidden)
        self.lin1 = torch.nn.Linear(hidden * 2, emb_dim)
        self.dropout = torch.nn.Dropout(dropout)

    def forward(self, x, edge_index, batch):
        x = F.relu(self.bn1(self.conv1(x, edge_index)))
        x = F.relu(self.bn2(self.conv2(x, edge_index)))
        x = F.relu(self.bn3(self.conv3(x, edge_index)))
        xm = global_mean_pool(x, batch); xM = global_max_pool(x, batch)
        cat = torch.cat([xm, xM], dim=1)
        emb = F.relu(self.lin1(cat)); emb = self.dropout(emb)
        return emb

    def extract_emb_batch(self, dataloader):
        self.eval()
        embs, labels = [], []
        with torch.no_grad():
            for data in dataloader:
                data = data.to(device)
                e = self.forward(data.x.float(), data.edge_index, data.batch)
                embs.append(e.cpu().numpy()); labels.append(data.y.cpu().numpy())
        return np.vstack(embs), np.concatenate(labels)

# ---------- helpers: training GIN (early stop on val loss) ----------
def train_gin(dataset, train_idx, val_idx, epochs=GIN_EPOCHS, patience=GIN_PATIENCE):
    train_ds = dataset[torch.tensor(train_idx)]
    val_ds = dataset[torch.tensor(val_idx)]
    train_loader = DataLoader(train_ds, batch_size=DL_BATCH_SIZE, shuffle=True)
    val_loader = DataLoader(val_ds, batch_size=DL_BATCH_SIZE)

    model = GINModel(dataset.num_features).to(device)
    head = torch.nn.Linear(GIN_EMB, dataset.num_classes).to(device)
    opt = torch.optim.Adam(list(model.parameters()) + list(head.parameters()), lr=3e-4, weight_decay=5e-5)
    crit = torch.nn.CrossEntropyLoss()

    best_val = float("inf"); wait = 0; best_state = None
    for ep in range(1, epochs+1):
        model.train(); head.train()
        tloss = 0.0
        for data in train_loader:
            data = data.to(device)
            opt.zero_grad()
            emb = model(data.x.float(), data.edge_index, data.batch)
            out = head(emb)
            loss = crit(out, data.y)
            loss.backward(); opt.step()
            tloss += loss.item() * data.num_graphs
        tloss /= len(train_loader.dataset)

        # val loss
        model.eval(); head.eval()
        vloss = 0.0
        with torch.no_grad():
            for data in val_loader:
                data = data.to(device)
                emb = model(data.x.float(), data.edge_index, data.batch)
                out = head(emb)
                vloss += crit(out, data.y).item() * data.num_graphs
        vloss /= len(val_loader.dataset) if len(val_loader.dataset) > 0 else 1.0

        if vloss < best_val - 1e-4:
            best_val = vloss; wait = 0
            best_state = (deepcopy(model.state_dict()), deepcopy(head.state_dict()))
        else:
            wait += 1
        if wait >= patience:
            break
    # load best
    if best_state is not None:
        model.load_state_dict(best_state[0]); head.load_state_dict(best_state[1])
    return model

# ---------- helpers: metrics ----------
def metrics_from_probs(probs, y_true):
    preds = np.argmax(probs, axis=1)
    return {
        "accuracy": accuracy_score(y_true, preds),
        "precision": precision_score(y_true, preds, zero_division=0),
        "recall": recall_score(y_true, preds, zero_division=0),
        "f1": f1_score(y_true, preds, zero_division=0),
        "roc_auc": roc_auc_score(y_true, probs[:,1]) if probs.shape[1] > 1 and len(np.unique(y_true))>1 else float("nan"),
        "log_loss": log_loss(y_true, probs)
    }

def logits_to_probs_binary(logits):
    arr = np.array(logits).reshape(-1)
    ppos = 1.0 / (1.0 + np.exp(-arr))
    probs = np.vstack([1-ppos, ppos]).T
    return probs

# ---------- Paillier inference (sample by sample) ----------
def paillier_inference(X, w, b, scale=PAILLIER_SCALE, key_size=PAILLIER_KEY):
    pk, sk = paillier.generate_paillier_keypair(n_length=key_size)
    w_int = [int(round(float(wi) * scale)) for wi in w.tolist()]
    b_int = int(round(float(b[0]) * (scale ** 2)))
    logits = []
    t0 = time.time()
    for x in X:
        x_int = [int(round(float(xj) * scale)) for xj in x.tolist()]
        enc_x = [pk.encrypt(int(xj)) for xj in x_int]
        acc = pk.encrypt(int(b_int))
        for enc_feat, wi in zip(enc_x, w_int):
            acc = acc + (enc_feat * int(wi))
        dec_int = sk.decrypt(acc)
        score = float(dec_int) / (scale ** 2)
        logits.append(score)
    wall = time.time() - t0
    return np.array(logits), wall

# ---------- CKKS single-sample ----------
def ckks_single_inference(X, w, b, ctx_scale_exp=CKKS_CTX_SCALE_EXP):
    ctx = ts.context(ts.SCHEME_TYPE.CKKS, poly_modulus_degree=8192, coeff_mod_bit_sizes=[60,40,40,60])
    ctx.global_scale = 2 ** ctx_scale_exp
    ctx.generate_galois_keys()
    enc_w = ts.ckks_vector(ctx, w.tolist())
    logits = []
    t0 = time.time()
    for x in X:
        enc_x = ts.ckks_vector(ctx, x.tolist())
        enc_dot = enc_x.dot(enc_w)
        dec_dot = enc_dot.decrypt()
        val = float(dec_dot[0]) if hasattr(dec_dot, "__len__") else float(dec_dot)
        val = val + float(b[0])
        ppos = 1.0 / (1.0 + np.exp(-val))
        eps = 1e-12
        logit = np.log(max(eps, ppos) / max(eps, 1 - ppos))
        logits.append(logit)
    wall = time.time() - t0
    return np.array(logits), wall

# ---------- CKKS batched inference ----------
def ckks_batched_inference(X, w, b, batch_size=CKKS_BATCH_SIZE, ctx_scale_exp=CKKS_CTX_SCALE_EXP):
    ctx = ts.context(ts.SCHEME_TYPE.CKKS, poly_modulus_degree=8192, coeff_mod_bit_sizes=[60,40,40,60])
    ctx.global_scale = 2 ** ctx_scale_exp
    ctx.generate_galois_keys()
    n = X.shape[0]
    logits = []
    t0 = time.time()
    for start in range(0, n, batch_size):
        end = min(start + batch_size, n)
        batch = X[start:end]
        bs = batch.shape[0]
        weighted_sum = None
        for j in range(batch.shape[1]):
            col_vals = batch[:, j].astype(float).tolist()
            enc_col = ts.ckks_vector(ctx, col_vals)
            term = enc_col * float(w[j])
            weighted_sum = term if weighted_sum is None else (weighted_sum + term)
        bias_vec = ts.ckks_vector(ctx, [float(b[0])] * bs)
        enc_logits = weighted_sum + bias_vec
        dec = enc_logits.decrypt()
        for v in dec:
            val = float(v) if np.isfinite(v) else 0.0
            ppos = 1.0 / (1.0 + np.exp(-val))
            eps = 1e-12
            logit = np.log(max(eps, ppos) / max(eps, 1 - ppos))
            logits.append(logit)
    wall = time.time() - t0
    return np.array(logits), wall

# ---------- main CV loop ----------
print("Loading dataset...")
dataset = TUDataset(root="/tmp/MUTAG_plain_he", name="MUTAG")
n = len(dataset)
labels = np.array([int(d.y.item()) for d in dataset])

skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=RND)

fold_results = []
fold = 0
for trainval_idx, test_idx in skf.split(np.arange(n), labels):
    fold += 1
    # split trainval into train and val for GIN early stop
    train_idx, val_idx = train_test_split(trainval_idx, test_size=0.125, random_state=RND, stratify=labels[trainval_idx])

    # train GIN
    gin = train_gin(dataset, train_idx, val_idx, epochs=GIN_EPOCHS, patience=GIN_PATIENCE)

    # extract embeddings
    train_loader = DataLoader(dataset[torch.tensor(train_idx)], batch_size=DL_BATCH_SIZE)
    val_loader   = DataLoader(dataset[torch.tensor(val_idx)], batch_size=DL_BATCH_SIZE)
    test_loader  = DataLoader(dataset[torch.tensor(test_idx)], batch_size=DL_BATCH_SIZE)

    X_train, y_train = gin.extract_emb_batch(train_loader)
    X_val,   y_val   = gin.extract_emb_batch(val_loader)
    X_test,  y_test  = gin.extract_emb_batch(test_loader)

    # train logistic regression (plaintext baseline)
    clf = LogisticRegression(max_iter=2000, solver="lbfgs")
    clf.fit(np.vstack([X_train, X_val]), np.concatenate([y_train, y_val]))

    # plaintext evaluation
    probs_plain = clf.predict_proba(X_test)
    metrics_plain = metrics_from_probs(probs_plain, y_test)
    logits_plain = clf.decision_function(X_test)

    # HE inference inputs (use positive-class weights for binary)
    w = clf.coef_.astype(float)
    b = clf.intercept_.astype(float)
    w_vec = w[0]

    # Paillier
    paillier_logits, paillier_time = paillier_inference(X_test, w_vec, b, scale=PAILLIER_SCALE, key_size=PAILLIER_KEY)
    probs_paillier = logits_to_probs_binary(paillier_logits)
    metrics_paillier = metrics_from_probs(probs_paillier, y_test)

    # CKKS single
    ckks_single_logits, ckks_single_time = ckks_single_inference(X_test, w_vec, b, ctx_scale_exp=CKKS_CTX_SCALE_EXP)
    probs_ckks_single = logits_to_probs_binary(ckks_single_logits)
    metrics_ckks_single = metrics_from_probs(probs_ckks_single, y_test)

    # CKKS batched
    ckks_batch_logits, ckks_batch_time = ckks_batched_inference(X_test, w_vec, b, batch_size=CKKS_BATCH_SIZE, ctx_scale_exp=CKKS_CTX_SCALE_EXP)
    probs_ckks_batch = logits_to_probs_binary(ckks_batch_logits)
    metrics_ckks_batch = metrics_from_probs(probs_ckks_batch, y_test)

    # Minimal per-fold printout
    print(f"\n=== Fold {fold}/{N_FOLDS} ===")
    print("train/val/test sizes:", len(train_idx), len(val_idx), len(test_idx))
    print(f"Plain accuracy: {metrics_plain['accuracy']:.4f}")

    fold_results.append({
        "plain": metrics_plain, "paillier": metrics_paillier,
        "ckks_single": metrics_ckks_single, "ckks_batch": metrics_ckks_batch,
        "time_paillier": paillier_time, "time_ckks_single": ckks_single_time, "time_ckks_batch": ckks_batch_time
    })

# ---------- aggregate results ----------
def agg(key):
    arr = [fr[key] for fr in fold_results]
    df = pd.DataFrame(arr)
    mean = df.mean().to_dict(); std = df.std().to_dict()
    return {k: (mean[k], std[k]) for k in mean.keys()}

print("\n=== Aggregated (mean ± std) across folds ===")
print("Plaintext:", {k: f"{v[0]:.4f} ± {v[1]:.4f}" for k,v in agg('plain').items()})
print("Paillier :", {k: f"{v[0]:.4f} ± {v[1]:.4f}" for k,v in agg('paillier').items()})
print("CKKS sing:", {k: f"{v[0]:.4f} ± {v[1]:.4f}" for k,v in agg('ckks_single').items()})
print("CKKS bat :", {k: f"{v[0]:.4f} ± {v[1]:.4f}" for k,v in agg('ckks_batch').items()})

# runtimes
times = pd.DataFrame([{"paillier":fr["time_paillier"], "ckks_single":fr["time_ckks_single"], "ckks_batch":fr["time_ckks_batch"]} for fr in fold_results])
print("\nAverage runtimes (s):", times.mean().to_dict()

Device: cpu
Loading dataset...


Processing...
Done!



=== Fold 1/3 ===
train/val/test sizes: 109 16 63
Plain accuracy: 0.8571

=== Fold 2/3 ===
train/val/test sizes: 109 16 63
Plain accuracy: 0.8571

=== Fold 3/3 ===
train/val/test sizes: 110 16 62
Plain accuracy: 0.8710

=== Aggregated (mean ± std) across folds ===
Plaintext: {'accuracy': '0.8618 ± 0.0080', 'precision': '0.8900 ± 0.0108', 'recall': '0.9040 ± 0.0013', 'f1': '0.8969 ± 0.0048', 'roc_auc': '0.9302 ± 0.0158', 'log_loss': '0.3681 ± 0.0149'}
Paillier : {'accuracy': '0.8618 ± 0.0080', 'precision': '0.8900 ± 0.0108', 'recall': '0.9040 ± 0.0013', 'f1': '0.8969 ± 0.0048', 'roc_auc': '0.9302 ± 0.0158', 'log_loss': '0.3681 ± 0.0149'}
CKKS sing: {'accuracy': '0.8618 ± 0.0080', 'precision': '0.8900 ± 0.0108', 'recall': '0.9040 ± 0.0013', 'f1': '0.8969 ± 0.0048', 'roc_auc': '0.9302 ± 0.0158', 'log_loss': '0.3681 ± 0.0149'}
CKKS bat : {'accuracy': '0.8618 ± 0.0080', 'precision': '0.8900 ± 0.0108', 'recall': '0.9040 ± 0.0013', 'f1': '0.8969 ± 0.0048', 'roc_auc': '0.9302 ± 0.0158', 'log_l

# **GraphSAGE**

In [ ]:
# =============================================================================
# Improved pipeline: GraphSAGE embeddings -> LogisticRegression -> HE inference
# Key changes:
#   - GraphSAGE with sum+mean+max pooling
#   - Longer training (60 epochs), patience=12
#   - CosineAnnealingLR scheduler
#   - Stronger weight decay
#   - Class-balanced LogisticRegression
# =============================================================================

import time, random
import numpy as np
import torch
import torch.nn.functional as F
from tqdm import tqdm
from copy import deepcopy
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, log_loss
from torch_geometric.datasets import TUDataset
from torch_geometric.loader import DataLoader
from torch_geometric.nn import SAGEConv, global_mean_pool, global_max_pool, global_add_pool
from torch.nn import BatchNorm1d

import phe as paillier
import tenseal as ts
import pandas as pd

# ---------------- USER KNOBS ----------------
RND = 42
N_FOLDS = 5
SAGE_HIDDEN = 128
SAGE_EMB = 256
SAGE_EPOCHS = 60
SAGE_PATIENCE = 12
PAILLIER_SCALE = 5000
PAILLIER_KEY = 2048
CKKS_CTX_SCALE_EXP = 40
CKKS_BATCH_SIZE = 8
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
# ---------------------------------------------

random.seed(RND); np.random.seed(RND); torch.manual_seed(RND)
device = torch.device(DEVICE)

# ---------- GraphSAGE model ----------
class SAGEModel(torch.nn.Module):
    def __init__(self, in_ch, hidden=SAGE_HIDDEN, emb_dim=SAGE_EMB, dropout=0.4):
        super().__init__()
        self.conv1 = SAGEConv(in_ch, hidden)
        self.bn1 = BatchNorm1d(hidden)
        self.conv2 = SAGEConv(hidden, hidden)
        self.bn2 = BatchNorm1d(hidden)
        self.conv3 = SAGEConv(hidden, hidden)
        self.bn3 = BatchNorm1d(hidden)

        # combine mean, max, sum pooling (hidden*3 -> emb_dim)
        self.lin1 = torch.nn.Linear(hidden * 3, emb_dim)
        self.lin2 = torch.nn.Linear(emb_dim, emb_dim)
        self.dropout = torch.nn.Dropout(dropout)

    def forward(self, x, edge_index, batch):
        x = F.relu(self.bn1(self.conv1(x, edge_index)))
        x = F.relu(self.bn2(self.conv2(x, edge_index)))
        x = F.relu(self.bn3(self.conv3(x, edge_index)))
        xm = global_mean_pool(x, batch)
        xM = global_max_pool(x, batch)
        xS = global_add_pool(x, batch)
        cat = torch.cat([xm, xM, xS], dim=1)
        emb = F.relu(self.lin1(cat))
        emb = self.dropout(F.relu(self.lin2(emb)))
        return emb

    def extract_emb_batch(self, dataloader):
        self.eval()
        embs, labels = [], []
        with torch.no_grad():
            for data in dataloader:
                data = data.to(device)
                e = self.forward(data.x.float(), data.edge_index, data.batch)
                embs.append(e.cpu().numpy()); labels.append(data.y.cpu().numpy())
        return np.vstack(embs), np.concatenate(labels)

# ---------- training helper ----------
def train_sage(dataset, train_idx, val_idx, epochs=SAGE_EPOCHS, patience=SAGE_PATIENCE):
    train_ds = dataset[torch.tensor(train_idx)]
    val_ds = dataset[torch.tensor(val_idx)]
    train_loader = DataLoader(train_ds, batch_size=32, shuffle=True)
    val_loader = DataLoader(val_ds, batch_size=32)

    model = SAGEModel(dataset.num_features).to(device)
    head = torch.nn.Linear(SAGE_EMB, dataset.num_classes).to(device)
    opt = torch.optim.Adam(list(model.parameters()) + list(head.parameters()), lr=5e-4, weight_decay=1e-4)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, epochs)
    crit = torch.nn.CrossEntropyLoss()

    best_val = float("inf"); wait = 0; best_state = None
    for ep in range(1, epochs+1):
        model.train(); head.train()
        for data in train_loader:
            data = data.to(device)
            opt.zero_grad()
            emb = model(data.x.float(), data.edge_index, data.batch)
            out = head(emb)
            loss = crit(out, data.y)
            loss.backward(); opt.step()
        sched.step()

        # val loss
        model.eval(); head.eval()
        vloss = 0.0
        with torch.no_grad():
            for data in val_loader:
                data = data.to(device)
                emb = model(data.x.float(), data.edge_index, data.batch)
                out = head(emb)
                vloss += crit(out, data.y).item() * data.num_graphs
        vloss /= len(val_loader.dataset)

        if vloss < best_val - 1e-4:
            best_val = vloss; wait = 0
            best_state = (deepcopy(model.state_dict()), deepcopy(head.state_dict()))
        else:
            wait += 1
        if wait >= patience:
            break

    if best_state is not None:
        model.load_state_dict(best_state[0]); head.load_state_dict(best_state[1])
    return model

# ---------- metrics ----------
def metrics_from_probs(probs, y_true):
    preds = np.argmax(probs, axis=1)
    return {
        "accuracy": accuracy_score(y_true, preds),
        "precision": precision_score(y_true, preds, zero_division=0),
        "recall": recall_score(y_true, preds, zero_division=0),
        "f1": f1_score(y_true, preds, zero_division=0),
        "roc_auc": roc_auc_score(y_true, probs[:,1]) if probs.shape[1] > 1 and len(np.unique(y_true))>1 else float("nan"),
        "log_loss": log_loss(y_true, probs)
    }

def logits_to_probs_binary(logits):
    arr = np.array(logits).reshape(-1)
    ppos = 1.0 / (1.0 + np.exp(-arr))
    probs = np.vstack([1-ppos, ppos]).T
    return probs

# ---------- Paillier inference ----------
def paillier_inference(X, w, b, scale=PAILLIER_SCALE, key_size=PAILLIER_KEY):
    pk, sk = paillier.generate_paillier_keypair(n_length=key_size)
    w_int = [int(round(float(wi) * scale)) for wi in w.tolist()]
    b_int = int(round(float(b[0]) * (scale ** 2)))
    logits = []
    t0 = time.time()
    for x in X:
        x_int = [int(round(float(xj) * scale)) for xj in x.tolist()]
        enc_x = [pk.encrypt(int(xj)) for xj in x_int]
        acc = pk.encrypt(int(b_int))
        for enc_feat, wi in zip(enc_x, w_int):
            acc = acc + (enc_feat * int(wi))
        dec_int = sk.decrypt(acc)
        score = float(dec_int) / (scale ** 2)
        logits.append(score)
    wall = time.time() - t0
    return np.array(logits), wall

# ---------- CKKS single ----------
def ckks_single_inference(X, w, b, ctx_scale_exp=CKKS_CTX_SCALE_EXP):
    ctx = ts.context(ts.SCHEME_TYPE.CKKS, poly_modulus_degree=8192, coeff_mod_bit_sizes=[60,40,40,60])
    ctx.global_scale = 2 ** ctx_scale_exp
    ctx.generate_galois_keys()
    enc_w = ts.ckks_vector(ctx, w.tolist())
    logits = []
    t0 = time.time()
    for x in X:
        enc_x = ts.ckks_vector(ctx, x.tolist())
        enc_dot = enc_x.dot(enc_w)
        dec_dot = enc_dot.decrypt()
        val = float(dec_dot[0]) if hasattr(dec_dot, "__len__") else float(dec_dot)
        val = val + float(b[0])
        ppos = 1.0 / (1.0 + np.exp(-val))
        eps = 1e-12
        logit = np.log(max(eps, ppos) / max(eps, 1 - ppos))
        logits.append(logit)
    wall = time.time() - t0
    return np.array(logits), wall

# ---------- CKKS batched ----------
def ckks_batched_inference(X, w, b, batch_size=CKKS_BATCH_SIZE, ctx_scale_exp=CKKS_CTX_SCALE_EXP):
    ctx = ts.context(ts.SCHEME_TYPE.CKKS, poly_modulus_degree=8192, coeff_mod_bit_sizes=[60,40,40,60])
    ctx.global_scale = 2 ** ctx_scale_exp
    ctx.generate_galois_keys()
    n = X.shape[0]
    logits = []
    t0 = time.time()
    for start in range(0, n, batch_size):
        end = min(start + batch_size, n)
        batch = X[start:end]
        bs = batch.shape[0]
        weighted_sum = None
        for j in range(batch.shape[1]):
            col_vals = batch[:, j].astype(float).tolist()
            enc_col = ts.ckks_vector(ctx, col_vals)
            term = enc_col * float(w[j])
            weighted_sum = term if weighted_sum is None else (weighted_sum + term)
        bias_vec = ts.ckks_vector(ctx, [float(b[0])] * bs)
        enc_logits = weighted_sum + bias_vec
        dec = enc_logits.decrypt()
        for v in dec:
            val = float(v) if np.isfinite(v) else 0.0
            ppos = 1.0 / (1.0 + np.exp(-val))
            eps = 1e-12
            logit = np.log(max(eps, ppos) / max(eps, 1 - ppos))
            logits.append(logit)
    wall = time.time() - t0
    return np.array(logits), wall

# ---------- main ----------
dataset = TUDataset(root="/tmp/MUTAG_plain_he", name="MUTAG")
n = len(dataset)
labels = np.array([int(d.y.item()) for d in dataset])

skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=RND)

fold_results = []
fold = 0
for trainval_idx, test_idx in skf.split(np.arange(n), labels):
    fold += 1
    train_idx, val_idx = train_test_split(trainval_idx, test_size=0.125, random_state=RND, stratify=labels[trainval_idx])

    sage = train_sage(dataset, train_idx, val_idx)

    train_loader = DataLoader(dataset[torch.tensor(train_idx)], batch_size=32)
    val_loader   = DataLoader(dataset[torch.tensor(val_idx)], batch_size=32)
    test_loader  = DataLoader(dataset[torch.tensor(test_idx)], batch_size=32)

    X_train, y_train = sage.extract_emb_batch(train_loader)
    X_val,   y_val   = sage.extract_emb_batch(val_loader)
    X_test,  y_test  = sage.extract_emb_batch(test_loader)

    clf = LogisticRegression(max_iter=5000, solver="lbfgs", class_weight="balanced")
    clf.fit(np.vstack([X_train, X_val]), np.concatenate([y_train, y_val]))

    probs_plain = clf.predict_proba(X_test)
    metrics_plain = metrics_from_probs(probs_plain, y_test)

    w = clf.coef_.astype(float)[0]
    b = clf.intercept_.astype(float)

    paillier_logits, paillier_time = paillier_inference(X_test, w, b)
    metrics_paillier = metrics_from_probs(logits_to_probs_binary(paillier_logits), y_test)

    ckks_single_logits, ckks_single_time = ckks_single_inference(X_test, w, b)
    metrics_ckks_single = metrics_from_probs(logits_to_probs_binary(ckks_single_logits), y_test)

    ckks_batch_logits, ckks_batch_time = ckks_batched_inference(X_test, w, b)
    metrics_ckks_batch = metrics_from_probs(logits_to_probs_binary(ckks_batch_logits), y_test)

    fold_results.append({
        "plain": metrics_plain, "paillier": metrics_paillier,
        "ckks_single": metrics_ckks_single, "ckks_batch": metrics_ckks_batch,
        "time_paillier": paillier_time, "time_ckks_single": ckks_single_time, "time_ckks_batch": ckks_batch_time
    })

# aggregate
import pandas as pd

def agg(key):
    arr = [fr[key] for fr in fold_results]
    df = pd.DataFrame(arr)
    mean = df.mean().to_dict(); std = df.std().to_dict()
    return {k: f"{mean[k]:.4f} ± {std[k]:.4f}" for k in mean.keys()}

print("\n=== Aggregated (mean ± std) ===")
print("Plaintext:", agg("plain"))
print("Paillier :", agg("paillier"))
print("CKKS sing:", agg("ckks_single"))
print("CKKS bat :", agg("ckks_batch"))

times = pd.DataFrame([{"paillier":fr["time_paillier"], "ckks_single":fr["time_ckks_single"], "ckks_batch":fr["time_ckks_batch"]} for fr in fold_results])
print("\nAverage runtimes (s):", times.mean().to_dict())



=== Aggregated (mean ± std) ===
Plaintext: {'accuracy': '0.8248 ± 0.0600', 'precision': '0.9165 ± 0.0567', 'recall': '0.8160 ± 0.1152', 'f1': '0.8578 ± 0.0604', 'roc_auc': '0.8837 ± 0.0262', 'log_loss': '0.5332 ± 0.1605'}
Paillier : {'accuracy': '0.8248 ± 0.0600', 'precision': '0.9165 ± 0.0567', 'recall': '0.8160 ± 0.1152', 'f1': '0.8578 ± 0.0604', 'roc_auc': '0.8837 ± 0.0262', 'log_loss': '0.5332 ± 0.1605'}
CKKS sing: {'accuracy': '0.8248 ± 0.0600', 'precision': '0.9165 ± 0.0567', 'recall': '0.8160 ± 0.1152', 'f1': '0.8578 ± 0.0604', 'roc_auc': '0.8837 ± 0.0262', 'log_loss': '0.5332 ± 0.1605'}
CKKS bat : {'accuracy': '0.8248 ± 0.0600', 'precision': '0.9165 ± 0.0567', 'recall': '0.8160 ± 0.1152', 'f1': '0.8578 ± 0.0604', 'roc_auc': '0.8837 ± 0.0262', 'log_loss': '0.5332 ± 0.1605'}

Average runtimes (s): {'paillier': 1153.1129339694976, 'ckks_single': 1.4109057903289794, 'ckks_batch': 14.672525978088379}


# **Graph Attention Model**

In [ ]:

import time, random
import numpy as np
import torch
import torch.nn.functional as F
from tqdm import tqdm
from copy import deepcopy
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, log_loss
from sklearn.preprocessing import StandardScaler
from torch_geometric.datasets import TUDataset
from torch_geometric.loader import DataLoader
from torch_geometric.nn import GATConv, global_mean_pool, global_max_pool, global_add_pool
from torch.nn import BatchNorm1d
import torch.nn as nn

import phe as paillier
import tenseal as ts
import pandas as pd

# ---------------- USER KNOBS ----------------
RND = 42
N_FOLDS = 5

# GAT params
GAT_HIDDEN = 256
GAT_EMB = 256
GAT_EPOCHS = 80
GAT_PATIENCE = 12
GAT_HEADS = 4
GAT_DROPOUT = 0.5

# Paillier tuning (faster, less secure than 2048)
PAILLIER_SCALE = 1000        # smaller quantization scale -> smaller big integers
PAILLIER_KEY = 1024          # reduced key size for speed (research/demo)
# CKKS tuning (faster, slightly lower precision)
CKKS_POLY = 4096
CKKS_CTX_SCALE_EXP = 20      # global_scale = 2**20
CKKS_BATCH_SIZE = 8

CKKS_COEFFS = [40, 20, 40]   # coefficient mod bit sizes suitable for poly=4096

LR_AUGMENT_COPIES = 2        # create small Gaussian-augmented duplicates of train embeddings
LR_AUG_NOISE_STD = 1e-2

PAILLIER_USE_REDUCED_KEY = True

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
# ---------------------------------------------

# reproducibility
random.seed(RND)
np.random.seed(RND)
torch.manual_seed(RND)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(RND)
device = torch.device(DEVICE)
print("Device:", device)

# ---------- GAT model ----------
class GATModel(torch.nn.Module):
    def __init__(self, in_ch, hidden=GAT_HIDDEN, emb_dim=GAT_EMB, heads=GAT_HEADS, dropout=GAT_DROPOUT):
        super().__init__()
        self.heads = heads
        self.hidden = hidden
        out_dim = hidden * heads

        self.conv1 = GATConv(in_ch, hidden, heads=heads, concat=True, dropout=dropout)
        self.bn1 = BatchNorm1d(out_dim)

        self.conv2 = GATConv(out_dim, hidden, heads=heads, concat=True, dropout=dropout)
        self.bn2 = BatchNorm1d(out_dim)

        self.conv3 = GATConv(out_dim, hidden, heads=heads, concat=True, dropout=dropout)
        self.bn3 = BatchNorm1d(out_dim)

        # after pooling mean+max+sum -> 3*out_dim
        self.lin1 = torch.nn.Linear(out_dim * 3, emb_dim)
        self.dropout = torch.nn.Dropout(dropout)

    def forward(self, x, edge_index, batch):
        # preserve a small projection for skip connection if input dim allows
        skip = x
        x = F.leaky_relu(self.bn1(self.conv1(x, edge_index)))
        x = F.leaky_relu(self.bn2(self.conv2(x, edge_index)))
        # safe skip-add: if dims mismatch, ignore skip
        if skip.shape[1] == x.shape[1]:
            x = x + skip
        x = F.leaky_relu(self.bn3(self.conv3(x, edge_index)))

        xm = global_mean_pool(x, batch)
        xM = global_max_pool(x, batch)
        xS = global_add_pool(x, batch)
        cat = torch.cat([xm, xM, xS], dim=1)

        emb = F.relu(self.lin1(cat))
        emb = self.dropout(emb)
        return emb

    def extract_emb_batch(self, dataloader):
        self.eval()
        embs, labels = [], []
        with torch.no_grad():
            for data in dataloader:
                data = data.to(device)
                e = self.forward(data.x.float(), data.edge_index, data.batch)
                embs.append(e.cpu().numpy()); labels.append(data.y.cpu().numpy())
        return np.vstack(embs), np.concatenate(labels)

# ---------- helpers: training GAT ----------
def train_gat(dataset, train_idx, val_idx, epochs=GAT_EPOCHS, patience=GAT_PATIENCE):
    train_ds = dataset[torch.tensor(train_idx)]
    val_ds = dataset[torch.tensor(val_idx)]
    train_loader = DataLoader(train_ds, batch_size=32, shuffle=True)
    val_loader = DataLoader(val_ds, batch_size=32)

    model = GATModel(dataset.num_features).to(device)
    head = torch.nn.Linear(GAT_EMB, dataset.num_classes).to(device)
    opt = torch.optim.AdamW(list(model.parameters()) + list(head.parameters()), lr=1e-3, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs)
    crit = torch.nn.CrossEntropyLoss()

    best_val = float("inf"); wait = 0; best_state = None
    for ep in range(1, epochs+1):
        model.train(); head.train()
        tloss = 0.0
        for data in train_loader:
            data = data.to(device)
            opt.zero_grad()
            emb = model(data.x.float(), data.edge_index, data.batch)
            out = head(emb)
            loss = crit(out, data.y)
            loss.backward(); opt.step()
            tloss += loss.item() * data.num_graphs
        tloss /= len(train_loader.dataset)

        # validation
        model.eval(); head.eval()
        vloss = 0.0
        with torch.no_grad():
            for data in val_loader:
                data = data.to(device)
                emb = model(data.x.float(), data.edge_index, data.batch)
                out = head(emb)
                vloss += crit(out, data.y).item() * data.num_graphs
        vloss /= len(val_loader.dataset)
        scheduler.step()

        # early stopping logic
        if vloss < best_val - 1e-4:
            best_val = vloss; wait = 0
            best_state = (deepcopy(model.state_dict()), deepcopy(head.state_dict()))
        else:
            wait += 1
        if wait >= patience:
            break

    # load best
    if best_state is not None:
        model.load_state_dict(best_state[0]); head.load_state_dict(best_state[1])
    return model

# ---------- metrics ----------
def metrics_from_probs(probs, y_true):
    preds = np.argmax(probs, axis=1)
    try:
        roc = roc_auc_score(y_true, probs[:,1])
    except:
        roc = float("nan")
    return {
        "accuracy": accuracy_score(y_true, preds),
        "precision": precision_score(y_true, preds, zero_division=0),
        "recall": recall_score(y_true, preds, zero_division=0),
        "f1": f1_score(y_true, preds, zero_division=0),
        "roc_auc": roc,
        "log_loss": log_loss(y_true, probs)
    }

def logits_to_probs_binary(logits):
    arr = np.array(logits).reshape(-1)
    ppos = 1.0 / (1.0 + np.exp(-arr))
    probs = np.vstack([1-ppos, ppos]).T
    return probs

# ---------- Paillier inference (improved: smaller key & quantized) ----------
def paillier_inference(X, w, b, scale=PAILLIER_SCALE, key_size=PAILLIER_KEY):
    """
    Encrypt each sample feature-wise using Paillier (phe).
    For speed we use a smaller key_size (1024 by default) and lower quantization scale.
    This remains exact arithmetic (no CKKS approx).
    """
    # generate keys once
    pk, sk = paillier.generate_paillier_keypair(n_length=key_size)
    # pre-scale weights and bias
    w_int = [int(round(float(wi) * scale)) for wi in w.tolist()]
    b_int = int(round(float(b[0]) * (scale ** 2)))  # bias scaled ~ scale^2 because x* w scaled twice

    logits = []
    t0 = time.time()
    # encrypt per-sample (phe doesn't support SIMD packing in this script)
    for x in X:
        # quantize input
        x_int = [int(round(float(xj) * scale)) for xj in x.tolist()]
        # encrypt features (vector of ciphertexts)
        enc_x = [pk.encrypt(int(xj)) for xj in x_int]
        # start accumulator with encrypted bias
        acc = pk.encrypt(int(b_int))
        # compute weighted sum in encrypted domain: acc += enc_x_j * w_int_j
        for enc_feat, wi in zip(enc_x, w_int):
            acc = acc + (enc_feat * int(wi))
        dec_int = sk.decrypt(acc)
        score = float(dec_int) / (scale ** 2)
        logits.append(score)
    wall = time.time() - t0
    return np.array(logits), wall

# ---------- CKKS single-sample (fast context & reuse) ----------
def ckks_single_inference(X, w, b, poly_mod=CKKS_POLY, ctx_scale_exp=CKKS_CTX_SCALE_EXP, coeff_mod_bit_sizes=CKKS_COEFFS):
    """
    For CKKS we build a context with poly_mod and global_scale,
    then encrypt each sample as CKKS vector and dot with plaintext weights.
    We reuse the same context and plaintext weight vector inside the function.
    """
    ctx = ts.context(ts.SCHEME_TYPE.CKKS, poly_modulus_degree=poly_mod, coeff_mod_bit_sizes=coeff_mod_bit_sizes)
    ctx.global_scale = 2 ** ctx_scale_exp
    ctx.generate_galois_keys()
    enc_w = ts.ckks_vector(ctx, w.tolist())   # plaintext weights as ckks vector (used in dot)
    logits = []
    t0 = time.time()
    for x in X:
        enc_x = ts.ckks_vector(ctx, x.tolist())   # encrypt sample vector
        enc_dot = enc_x.dot(enc_w)                # encrypted dot product (CKKS)
        dec_dot = enc_dot.decrypt()
        # decrypt returns scalar-like or length-1 list depending on version
        val = float(dec_dot[0]) if hasattr(dec_dot, "__len__") else float(dec_dot)
        val = val + float(b[0])
        # stable sigmoid -> logit
        ppos = 1.0 / (1.0 + np.exp(-val))
        eps = 1e-12
        logit = np.log(max(eps, ppos) / max(eps, 1 - ppos))
        logits.append(logit)
    wall = time.time() - t0
    return np.array(logits), wall

# ---------- CKKS batched inference (encrypt columns across batch OR encrypt each row but reuse context) ----------
def ckks_batched_inference(X, w, b, batch_size=CKKS_BATCH_SIZE, poly_mod=CKKS_POLY, ctx_scale_exp=CKKS_CTX_SCALE_EXP, coeff_mod_bit_sizes=CKKS_COEFFS):
    """
    Batched CKKS inference: for each small batch, encrypt each sample vector and compute dot(enc_x, enc_w)
    This is straightforward and benefits from smaller poly_mod and scale; it's fast in practice.
    """
    ctx = ts.context(ts.SCHEME_TYPE.CKKS, poly_modulus_degree=poly_mod, coeff_mod_bit_sizes=coeff_mod_bit_sizes)
    ctx.global_scale = 2 ** ctx_scale_exp
    ctx.generate_galois_keys()
    enc_w = ts.ckks_vector(ctx, w.tolist())   # plaintext weights as ckks vector
    n = X.shape[0]
    logits = []
    t0 = time.time()
    for start in range(0, n, batch_size):
        end = min(start + batch_size, n)
        batch = X[start:end]    # shape (bs, d)
        # encrypt each sample in batch and compute dot
        enc_batch_logits = []
        for x in batch:
            enc_x = ts.ckks_vector(ctx, x.tolist())
            enc_dot = enc_x.dot(enc_w)
            enc_batch_logits.append(enc_dot)
        # decrypt all batch results
        for enc_dot in enc_batch_logits:
            dec_dot = enc_dot.decrypt()
            val = float(dec_dot[0]) if hasattr(dec_dot, "__len__") else float(dec_dot)
            val = val + float(b[0])
            ppos = 1.0 / (1.0 + np.exp(-val))
            eps = 1e-12
            logit = np.log(max(eps, ppos) / max(eps, 1 - ppos))
            logits.append(logit)
    wall = time.time() - t0
    return np.array(logits), wall

# ---------- main CV loop ----------
print("Loading dataset...")
dataset = TUDataset(root="/tmp/MUTAG_plain_he", name="MUTAG")
n = len(dataset)
labels = np.array([int(d.y.item()) for d in dataset])

skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=RND)

fold_results = []
fold = 0
for trainval_idx, test_idx in skf.split(np.arange(n), labels):
    fold += 1
    print(f"\n=== Fold {fold}/{N_FOLDS} ===")
    # split trainval into train and val for GAT early stop
    train_idx, val_idx = train_test_split(trainval_idx, test_size=0.125, random_state=RND, stratify=labels[trainval_idx])
    print("train/val/test sizes:", len(train_idx), len(val_idx), len(test_idx))

    # 1) train GAT
    gat = train_gat(dataset, train_idx, val_idx, epochs=GAT_EPOCHS, patience=GAT_PATIENCE)

    # 2) extract embeddings for train, val, test
    train_loader = DataLoader(dataset[torch.tensor(train_idx)], batch_size=32)
    val_loader   = DataLoader(dataset[torch.tensor(val_idx)], batch_size=32)
    test_loader  = DataLoader(dataset[torch.tensor(test_idx)], batch_size=32)

    X_train, y_train = gat.extract_emb_batch(train_loader)
    X_val,   y_val   = gat.extract_emb_batch(val_loader)
    X_test,  y_test  = gat.extract_emb_batch(test_loader)
    print("Emb shapes (train/val/test):", X_train.shape, X_val.shape, X_test.shape)

    # 3) scale embeddings (StandardScaler) and augmentation for LR
    scaler = StandardScaler()
    scaler.fit(np.vstack([X_train, X_val]))
    X_train_s = scaler.transform(X_train)
    X_val_s = scaler.transform(X_val)
    X_test_s = scaler.transform(X_test)

    # augmentation: duplicate train+val examples with small gaussian noise
    def augment(X, y, copies=LR_AUGMENT_COPIES, noise_std=LR_AUG_NOISE_STD):
        Xs, ys = [X], [y]
        for _ in range(copies):
            Xs.append(X + np.random.normal(scale=noise_std, size=X.shape))
            ys.append(y.copy())
        return np.vstack(Xs), np.concatenate(ys)
    X_aug, y_aug = augment(np.vstack([X_train_s, X_val_s]), np.concatenate([y_train, y_val]), copies=LR_AUGMENT_COPIES, noise_std=LR_AUG_NOISE_STD)

    # 4) train logistic regression
    clf = LogisticRegression(max_iter=10000, solver="lbfgs", class_weight="balanced")
    clf.fit(X_aug, y_aug)
    probs_plain = clf.predict_proba(X_test_s)
    metrics_plain = metrics_from_probs(probs_plain, y_test)

    # 5) prepare HE inference (work with raw logits from clf on scaled embeddings)
    logits_plain = clf.decision_function(X_test_s)  # raw logits

    w = clf.coef_.astype(float)   # shape (1, d) for binary
    b = clf.intercept_.astype(float)
    w_vec = w[0]

    # 6) Paillier inference (quantized, smaller key)
    paillier_logits, paillier_time = paillier_inference(X_test_s, w_vec, b, scale=PAILLIER_SCALE, key_size=PAILLIER_KEY)
    probs_paillier = logits_to_probs_binary(paillier_logits)
    metrics_paillier = metrics_from_probs(probs_paillier, y_test)

    # 7) CKKS single-sample (fast context)
    ckks_single_logits, ckks_single_time = ckks_single_inference(X_test_s, w_vec, b, poly_mod=CKKS_POLY, ctx_scale_exp=CKKS_CTX_SCALE_EXP, coeff_mod_bit_sizes=CKKS_COEFFS)
    probs_ckks_single = logits_to_probs_binary(ckks_single_logits)
    metrics_ckks_single = metrics_from_probs(probs_ckks_single, y_test)

    # 8) CKKS batched
    ckks_batch_logits, ckks_batch_time = ckks_batched_inference(X_test_s, w_vec, b, batch_size=CKKS_BATCH_SIZE, poly_mod=CKKS_POLY, ctx_scale_exp=CKKS_CTX_SCALE_EXP, coeff_mod_bit_sizes=CKKS_COEFFS)
    probs_ckks_batch = logits_to_probs_binary(ckks_batch_logits)
    metrics_ckks_batch = metrics_from_probs(probs_ckks_batch, y_test)

    print("Fold results (Plain / Paillier / CKKS single / CKKS batched):")
    print(metrics_plain)
    print(metrics_paillier)
    print(metrics_ckks_single)
    print(metrics_ckks_batch)
    print("Runtimes (s): paillier=%.2f, ckks_single=%.2f, ckks_batch=%.2f" % (paillier_time, ckks_single_time, ckks_batch_time))

    fold_results.append({
        "plain": metrics_plain, "paillier": metrics_paillier,
        "ckks_single": metrics_ckks_single, "ckks_batch": metrics_ckks_batch,
        "time_paillier": paillier_time, "time_ckks_single": ckks_single_time, "time_ckks_batch": ckks_batch_time
    })

# ---------- aggregate results ----------
def agg(key):
    arr = [fr[key] for fr in fold_results]
    df = pd.DataFrame(arr)
    mean = df.mean().to_dict(); std = df.std().to_dict()
    return {k: (mean[k], std[k]) for k in mean.keys()}

print("\n=== Aggregated (mean ± std) across folds ===")
print("Plaintext:", {k: f"{v[0]:.4f} ± {v[1]:.4f}" for k,v in agg('plain').items()})
print("Paillier :", {k: f"{v[0]:.4f} ± {v[1]:.4f}" for k,v in agg('paillier').items()})
print("CKKS sing:", {k: f"{v[0]:.4f} ± {v[1]:.4f}" for k,v in agg('ckks_single').items()})
print("CKKS bat :", {k: f"{v[0]:.4f} ± {v[1]:.4f}" for k,v in agg('ckks_batch').items()})

# runtimes summary
times = pd.DataFrame([{"paillier":fr["time_paillier"], "ckks_single":fr["time_ckks_single"], "ckks_batch":fr["time_ckks_batch"]} for fr in fold_results])
print("\nAverage runtimes (s):", times.mean().to_dict())

print("\nNotes:")
print("- Paillier uses 1024-bit keys and smaller quantization scale (faster, less secure than 2048).")
print("- CKKS uses poly_modulus_degree=%d and global_scale=2**%d for a large speedup." % (CKKS_POLY, CKKS_CTX_SCALE_EXP))
print("- We standardized embeddings and added small augmentation copies for LR to reduce overfitting.")
print("- If you want even faster Paillier, next step is integer-packing of multiple features into one ciphertext (more complex).")


Device: cpu
Loading dataset...

=== Fold 1/5 ===
train/val/test sizes: 131 19 38
Emb shapes (train/val/test): (131, 256) (19, 256) (38, 256)
Fold results (Plain / Paillier / CKKS single / CKKS batched):
{'accuracy': 0.868421052631579, 'precision': 0.8571428571428571, 'recall': 0.96, 'f1': 0.9056603773584906, 'roc_auc': np.float64(0.9230769230769231), 'log_loss': 0.33811013730195466}
{'accuracy': 0.868421052631579, 'precision': 0.8571428571428571, 'recall': 0.96, 'f1': 0.9056603773584906, 'roc_auc': np.float64(0.9230769230769231), 'log_loss': 0.33815869970822243}
{'accuracy': 0.8157894736842105, 'precision': 0.875, 'recall': 0.84, 'f1': 0.8571428571428571, 'roc_auc': np.float64(0.92), 'log_loss': 0.3276279529499653}
{'accuracy': 0.8157894736842105, 'precision': 0.8, 'recall': 0.96, 'f1': 0.8727272727272727, 'roc_auc': np.float64(0.9199999999999999), 'log_loss': 0.3526206340202893}
Runtimes (s): paillier=169.42, ckks_single=0.57, ckks_batch=0.37

=== Fold 2/5 ===
train/val/test sizes: 13

# **GCN Model**

In [ ]:

import time, random, math
import numpy as np
import torch
import torch.nn.functional as F
from tqdm import tqdm
from copy import deepcopy
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.linear_model import LogisticRegressionCV
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, log_loss
from sklearn.preprocessing import StandardScaler
from torch_geometric.datasets import TUDataset
from torch_geometric.loader import DataLoader
from torch_geometric.nn import GCNConv, global_mean_pool, global_max_pool, global_add_pool
from torch.nn import BatchNorm1d, LayerNorm
import torch.nn as nn

import phe as paillier
import tenseal as ts
import pandas as pd

# ---------------- USER KNOBS ----------------
RND = 42
N_FOLDS = 5

# GCN params
GCN_HIDDEN = 256
GCN_EMB = 256
GCN_EPOCHS = 120
GCN_PATIENCE = 16
GCN_DROPOUT = 0.5
GCN_LR = 1e-3
GCN_WEIGHT_DECAY = 1e-4
GCN_GRAD_CLIP = 5.0
GCN_FEATURE_DROPOUT = 0.05

# Paillier tuning
PAILLIER_SCALE = 1000
PAILLIER_KEY = 1024

# CKKS tuning
CKKS_POLY = 4096
CKKS_CTX_SCALE_EXP = 20
CKKS_BATCH_SIZE = 8
CKKS_COEFFS = [40, 20, 40]

LR_AUGMENT_COPIES = 2
LR_AUG_NOISE_STD_REL = 0.01

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
device = torch.device(DEVICE)
print("Device:", device)

# ---------------- reproducibility ----------------
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(RND)

# ---------- improved GCN model ----------
class GCNModel(torch.nn.Module):
    def __init__(self, in_ch, hidden=GCN_HIDDEN, emb_dim=GCN_EMB, dropout=GCN_DROPOUT, n_layers=3):
        super().__init__()
        self.n_layers = n_layers
        self.convs = nn.ModuleList()
        self.norms = nn.ModuleList()

        self.convs.append(GCNConv(in_ch, hidden))
        self.norms.append(BatchNorm1d(hidden))

        for _ in range(1, n_layers):
            self.convs.append(GCNConv(hidden, hidden))
            self.norms.append(LayerNorm(hidden))

        self.lin1 = nn.Linear(hidden * 3, emb_dim)
        self.dropout = nn.Dropout(dropout)
        self._init_weights()

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.xavier_uniform_(m.weight)
                if m.bias is not None:
                    nn.init.zeros_(m.bias)
            if isinstance(m, GCNConv):
                try:
                    if hasattr(m, 'lin') and hasattr(m.lin, 'weight'):
                        nn.init.xavier_uniform_(m.lin.weight)
                except Exception:
                    pass

    def forward(self, x, edge_index, batch, feat_dropout=0.0):
        if self.training and feat_dropout > 0:
            mask = (torch.rand_like(x) > feat_dropout).float()
            x = x * mask

        residual = None
        for i in range(self.n_layers):
            h = self.convs[i](x, edge_index)
            h = self.norms[i](h)
            h = F.gelu(h)
            if residual is not None and residual.shape == h.shape:
                h = h + residual
            residual = h
            x = F.dropout(h, p=self.dropout.p if isinstance(self.dropout, nn.Dropout) else 0.5, training=self.training)

        xm = global_mean_pool(x, batch)
        xM = global_max_pool(x, batch)
        xS = global_add_pool(x, batch)
        cat = torch.cat([xm, xM, xS], dim=1)

        emb = F.gelu(self.lin1(cat))
        emb = self.dropout(emb)
        return emb

    def extract_emb_batch(self, dataloader):
        self.eval()
        embs, labels = [], []
        with torch.no_grad():
            for data in dataloader:
                data = data.to(device)
                e = self.forward(data.x.float(), data.edge_index, data.batch, feat_dropout=0.0)
                embs.append(e.cpu().numpy()); labels.append(data.y.cpu().numpy())
        return np.vstack(embs), np.concatenate(labels)

# ---------- helpers: training GCN ----------
def train_gcn(dataset, train_idx, val_idx, epochs=GCN_EPOCHS, patience=GCN_PATIENCE, lr=GCN_LR, weight_decay=GCN_WEIGHT_DECAY):
    train_ds = dataset[torch.tensor(train_idx)]
    val_ds = dataset[torch.tensor(val_idx)]
    train_loader = DataLoader(train_ds, batch_size=32, shuffle=True)
    val_loader = DataLoader(val_ds, batch_size=32)

    model = GCNModel(dataset.num_features).to(device)
    head = torch.nn.Linear(GCN_EMB, dataset.num_classes).to(device)

    opt = torch.optim.AdamW(list(model.parameters()) + list(head.parameters()), lr=lr, weight_decay=weight_decay)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, mode='min', factor=0.5, patience=6, min_lr=1e-6)
    crit = torch.nn.CrossEntropyLoss()

    best_val = float("inf"); wait = 0; best_state = None
    for ep in range(1, epochs+1):
        model.train(); head.train()
        tloss = 0.0
        for data in train_loader:
            data = data.to(device)
            opt.zero_grad()
            emb = model(data.x.float(), data.edge_index, data.batch, feat_dropout=GCN_FEATURE_DROPOUT)
            out = head(emb)
            loss = crit(out, data.y)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(list(model.parameters()) + list(head.parameters()), max_norm=GCN_GRAD_CLIP)
            opt.step()
            tloss += loss.item() * data.num_graphs
        tloss /= len(train_loader.dataset)

        model.eval(); head.eval()
        vloss = 0.0
        with torch.no_grad():
            for data in val_loader:
                data = data.to(device)
                emb = model(data.x.float(), data.edge_index, data.batch, feat_dropout=0.0)
                out = head(emb)
                vloss += crit(out, data.y).item() * data.num_graphs
        vloss /= len(val_loader.dataset)

        scheduler.step(vloss)

        if vloss < best_val - 1e-4:
            best_val = vloss; wait = 0
            best_state = (deepcopy(model.state_dict()), deepcopy(head.state_dict()))
        else:
            wait += 1

        if ep % 5 == 0 or ep == 1:
            print(f"Epoch {ep:03d} — train_loss: {tloss:.4f}, val_loss: {vloss:.4f}, lr: {opt.param_groups[0]['lr']:.6f}, wait: {wait}")

        if wait >= patience:
            print(f"Early stopping at epoch {ep} (wait={wait} >= patience={patience})")
            break

    if best_state is not None:
        model.load_state_dict(best_state[0]); head.load_state_dict(best_state[1])
    return model

# ---------- metrics ----------
def metrics_from_probs(probs, y_true):
    preds = np.argmax(probs, axis=1)
    try:
        roc = roc_auc_score(y_true, probs[:,1])
    except Exception:
        roc = float("nan")
    return {
        "accuracy": accuracy_score(y_true, preds),
        "precision": precision_score(y_true, preds, zero_division=0),
        "recall": recall_score(y_true, preds, zero_division=0),
        "f1": f1_score(y_true, preds, zero_division=0),
        "roc_auc": roc,
        "log_loss": log_loss(y_true, probs)
    }

def logits_to_probs_binary(logits):
    arr = np.array(logits).reshape(-1)
    ppos = 1.0 / (1.0 + np.exp(-arr))
    probs = np.vstack([1-ppos, ppos]).T
    return probs

# ---------- Paillier inference ----------
def paillier_inference(X, w, b, scale=PAILLIER_SCALE, key_size=PAILLIER_KEY):
    pk, sk = paillier.generate_paillier_keypair(n_length=key_size)
    w_int = [int(round(float(wi) * scale)) for wi in w.tolist()]
    b_int = int(round(float(b) * (scale ** 2)))

    logits = []
    t0 = time.time()
    for x in X:
        x_int = [int(round(float(xj) * scale)) for xj in x.tolist()]
        enc_x = [pk.encrypt(int(xj)) for xj in x_int]
        acc = pk.encrypt(int(b_int))
        for enc_feat, wi in zip(enc_x, w_int):
            acc = acc + (enc_feat * int(wi))
        dec_int = sk.decrypt(acc)
        score = float(dec_int) / (scale ** 2)
        logits.append(score)
    wall = time.time() - t0
    return np.array(logits), wall

# ---------- CKKS single inference ----------
def ckks_single_inference(X, w, b, poly_mod=CKKS_POLY, ctx_scale_exp=CKKS_CTX_SCALE_EXP, coeff_mod_bit_sizes=CKKS_COEFFS):
    ctx = ts.context(ts.SCHEME_TYPE.CKKS, poly_modulus_degree=poly_mod, coeff_mod_bit_sizes=coeff_mod_bit_sizes)
    ctx.global_scale = 2 ** ctx_scale_exp
    ctx.generate_galois_keys()
    enc_w = ts.ckks_vector(ctx, w.tolist())
    logits = []
    t0 = time.time()
    for x in X:
        enc_x = ts.ckks_vector(ctx, x.tolist())
        enc_dot = enc_x.dot(enc_w)
        dec_dot = enc_dot.decrypt()
        val = float(dec_dot[0]) if hasattr(dec_dot, "__len__") else float(dec_dot)
        val = val + float(b)
        ppos = 1.0 / (1.0 + np.exp(-val))
        eps = 1e-12
        logit = math.log(max(eps, ppos) / max(eps, 1 - ppos))
        logits.append(logit)
    wall = time.time() - t0
    return np.array(logits), wall

# ---------- CKKS batched inference ----------
def ckks_batched_inference(X, w, b, batch_size=CKKS_BATCH_SIZE, poly_mod=CKKS_POLY, ctx_scale_exp=CKKS_CTX_SCALE_EXP, coeff_mod_bit_sizes=CKKS_COEFFS):
    ctx = ts.context(ts.SCHEME_TYPE.CKKS, poly_modulus_degree=poly_mod, coeff_mod_bit_sizes=coeff_mod_bit_sizes)
    ctx.global_scale = 2 ** ctx_scale_exp
    ctx.generate_galois_keys()
    enc_w = ts.ckks_vector(ctx, w.tolist())
    n = X.shape[0]
    logits = []
    t0 = time.time()
    for start in range(0, n, batch_size):
        end = min(start + batch_size, n)
        batch = X[start:end]
        enc_batch = [ts.ckks_vector(ctx, x.tolist()) for x in batch]
        enc_batch_dots = [enc_x.dot(enc_w) for enc_x in enc_batch]
        for enc_dot in enc_batch_dots:
            dec_dot = enc_dot.decrypt()
            val = float(dec_dot[0]) if hasattr(dec_dot, "__len__") else float(dec_dot)
            val = val + float(b)
            ppos = 1.0 / (1.0 + np.exp(-val))
            eps = 1e-12
            logit = math.log(max(eps, ppos) / max(eps, 1 - ppos))
            logits.append(logit)
    wall = time.time() - t0
    return np.array(logits), wall

# ---------- main CV loop ----------
print("Loading dataset...")
dataset = TUDataset(root="/tmp/MUTAG_plain_he", name="MUTAG")
n = len(dataset)
labels = np.array([int(d.y.item()) for d in dataset])

skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=RND)

fold_results = []
fold = 0
for trainval_idx, test_idx in skf.split(np.arange(n), labels):
    fold += 1
    print(f"\n=== Fold {fold}/{N_FOLDS} ===")
    train_idx, val_idx = train_test_split(trainval_idx, test_size=0.125, random_state=RND, stratify=labels[trainval_idx])
    print("train/val/test sizes:", len(train_idx), len(val_idx), len(test_idx))

    gcn = train_gcn(dataset, train_idx, val_idx, epochs=GCN_EPOCHS, patience=GCN_PATIENCE)

    train_loader = DataLoader(dataset[torch.tensor(train_idx)], batch_size=32)
    val_loader   = DataLoader(dataset[torch.tensor(val_idx)], batch_size=32)
    test_loader  = DataLoader(dataset[torch.tensor(test_idx)], batch_size=32)

    X_train, y_train = gcn.extract_emb_batch(train_loader)
    X_val,   y_val   = gcn.extract_emb_batch(val_loader)
    X_test,  y_test  = gcn.extract_emb_batch(test_loader)
    print("Emb shapes (train/val/test):", X_train.shape, X_val.shape, X_test.shape)

    scaler = StandardScaler()
    scaler.fit(np.vstack([X_train, X_val]))
    X_train_s = scaler.transform(X_train)
    X_val_s = scaler.transform(X_val)
    X_test_s = scaler.transform(X_test)

    def augment(X, y, copies=LR_AUGMENT_COPIES, noise_std_rel=LR_AUG_NOISE_STD_REL):
        Xs, ys = [X], [y]
        feat_std = np.std(X, axis=0, ddof=0)
        feat_std = np.where(feat_std == 0.0, 1.0, feat_std)
        for _ in range(copies):
            noise = np.random.normal(scale=noise_std_rel * feat_std, size=X.shape)
            Xs.append(X + noise)
            ys.append(y.copy())
        return np.vstack(Xs), np.concatenate(ys)

    X_aug, y_aug = augment(np.vstack([X_train_s, X_val_s]), np.concatenate([y_train, y_val]))

    clf = LogisticRegressionCV(
        Cs=5,
        cv=3,
        penalty='l2',
        scoring='f1',
        solver='lbfgs',
        max_iter=10000,
        class_weight='balanced',
        n_jobs=1,
        refit=True
    )
    clf.fit(X_aug, y_aug)

    probs_plain = clf.predict_proba(X_test_s)
    metrics_plain = metrics_from_probs(probs_plain, y_test)

    logits_plain = clf.decision_function(X_test_s)
    w = clf.coef_.astype(float)
    b = clf.intercept_.astype(float)
    w_vec = w[0]
    b_scalar = float(b[0]) if hasattr(b, "__len__") else float(b)

    paillier_logits, paillier_time = paillier_inference(X_test_s, w_vec, b_scalar)
    probs_paillier = logits_to_probs_binary(paillier_logits)
    metrics_paillier = metrics_from_probs(probs_paillier, y_test)

    ckks_single_logits, ckks_single_time = ckks_single_inference(X_test_s, w_vec, b_scalar)
    probs_ckks_single = logits_to_probs_binary(ckks_single_logits)
    metrics_ckks_single = metrics_from_probs(probs_ckks_single, y_test)

    ckks_batch_logits, ckks_batch_time = ckks_batched_inference(X_test_s, w_vec, b_scalar)
    probs_ckks_batch = logits_to_probs_binary(ckks_batch_logits)
    metrics_ckks_batch = metrics_from_probs(probs_ckks_batch, y_test)

    print("Fold results (Plain / Paillier / CKKS single / CKKS batched):")
    print("Plain:", metrics_plain)
    print("Paillier:", metrics_paillier)
    print("CKKS single:", metrics_ckks_single)
    print("CKKS batched:", metrics_ckks_batch)
    print("Runtimes (s): paillier=%.2f, ckks_single=%.2f, ckks_batch=%.2f" % (paillier_time, ckks_single_time, ckks_batch_time))

    fold_results.append({
        "plain": metrics_plain, "paillier": metrics_paillier,
        "ckks_single": metrics_ckks_single, "ckks_batch": metrics_ckks_batch,
        "time_paillier": paillier_time, "time_ckks_single": ckks_single_time, "time_ckks_batch": ckks_batch_time
    })

# ---------- aggregate results ----------
def agg(key):
    arr = [fr[key] for fr in fold_results]
    df = pd.DataFrame


Device: cpu
Loading dataset...

=== Fold 1/5 ===
train/val/test sizes: 131 19 38
Epoch 001 — train_loss: 4.9154, val_loss: 0.5111, lr: 0.001000, wait: 0
Epoch 005 — train_loss: 1.5353, val_loss: 0.4184, lr: 0.001000, wait: 0
Epoch 010 — train_loss: 0.8528, val_loss: 0.8669, lr: 0.001000, wait: 3
Epoch 015 — train_loss: 0.4735, val_loss: 0.4975, lr: 0.001000, wait: 1
Epoch 020 — train_loss: 0.5362, val_loss: 0.4691, lr: 0.001000, wait: 6
Epoch 025 — train_loss: 0.4961, val_loss: 0.4319, lr: 0.000500, wait: 11
Epoch 030 — train_loss: 0.3984, val_loss: 0.4901, lr: 0.000250, wait: 16
Early stopping at epoch 30 (wait=16 >= patience=16)
Emb shapes (train/val/test): (131, 256) (19, 256) (38, 256)
Fold results (Plain / Paillier / CKKS single / CKKS batched):
Plain: {'accuracy': 0.8157894736842105, 'precision': 0.78125, 'recall': 1.0, 'f1': 0.8771929824561403, 'roc_auc': np.float64(0.8261538461538461), 'log_loss': 2.832243580536838}
Paillier: {'accuracy': 0.8157894736842105, 'precision': 0.7812

/tmp/ipython-input-619941885.py:213: RuntimeWarning: overflow encountered in exp
  ppos = 1.0 / (1.0 + np.exp(-arr))
/tmp/ipython-input-619941885.py:251: RuntimeWarning: overflow encountered in exp
  ppos = 1.0 / (1.0 + np.exp(-val))
/tmp/ipython-input-619941885.py:276: RuntimeWarning: overflow encountered in exp
  ppos = 1.0 / (1.0 + np.exp(-val))


Fold results (Plain / Paillier / CKKS single / CKKS batched):
Plain: {'accuracy': 0.7631578947368421, 'precision': 0.9444444444444444, 'recall': 0.68, 'f1': 0.7906976744186046, 'roc_auc': np.float64(0.8707692307692307), 'log_loss': 2.239479414918353}
Paillier: {'accuracy': 0.7631578947368421, 'precision': 0.9444444444444444, 'recall': 0.68, 'f1': 0.7906976744186046, 'roc_auc': np.float64(0.8676923076923078), 'log_loss': 2.2335262095853308}
CKKS single: {'accuracy': 0.7894736842105263, 'precision': 1.0, 'recall': 0.68, 'f1': 0.8095238095238095, 'roc_auc': np.float64(0.8692307692307693), 'log_loss': 2.068597597120801}
CKKS batched: {'accuracy': 0.7631578947368421, 'precision': 0.9444444444444444, 'recall': 0.68, 'f1': 0.7906976744186046, 'roc_auc': np.float64(0.8692307692307693), 'log_loss': 1.9747261133995158}
Runtimes (s): paillier=175.22, ckks_single=0.39, ckks_batch=0.38

=== Fold 3/5 ===
train/val/test sizes: 131 19 38
Epoch 001 — train_loss: 6.7478, val_loss: 1.7172, lr: 0.001000, 

/tmp/ipython-input-619941885.py:213: RuntimeWarning: overflow encountered in exp
  ppos = 1.0 / (1.0 + np.exp(-arr))
/tmp/ipython-input-619941885.py:251: RuntimeWarning: overflow encountered in exp
  ppos = 1.0 / (1.0 + np.exp(-val))


Fold results (Plain / Paillier / CKKS single / CKKS batched):
Plain: {'accuracy': 0.8378378378378378, 'precision': 0.9130434782608695, 'recall': 0.84, 'f1': 0.875, 'roc_auc': np.float64(0.8833333333333333), 'log_loss': 0.8500835966710165}
Paillier: {'accuracy': 0.8378378378378378, 'precision': 0.9130434782608695, 'recall': 0.84, 'f1': 0.875, 'roc_auc': np.float64(0.8833333333333333), 'log_loss': 0.8494818745059481}
CKKS single: {'accuracy': 0.8378378378378378, 'precision': 0.9523809523809523, 'recall': 0.8, 'f1': 0.8695652173913043, 'roc_auc': np.float64(0.8866666666666667), 'log_loss': 0.9388361854271368}
CKKS batched: {'accuracy': 0.8108108108108109, 'precision': 0.9090909090909091, 'recall': 0.8, 'f1': 0.851063829787234, 'roc_auc': np.float64(0.88), 'log_loss': 0.8749635417575153}
Runtimes (s): paillier=170.51, ckks_single=0.37, ckks_batch=0.36


/tmp/ipython-input-619941885.py:276: RuntimeWarning: overflow encountered in exp
  ppos = 1.0 / (1.0 + np.exp(-val))


# **Proposed HybridGNN MuTag**

In [10]:
# =============================================================================
# Improved Hybrid GIN+GAT -> LogisticRegression -> HE pipeline (MUTAG)
# - Hybrid encoder (GIN + GAT) with residuals + LayerNorm + richer pooling
# - AdamW + CosineAnnealingLR, label smoothing, DropEdge (optional)
# - StandardScaler + optional PCA before HE
# - LogisticRegression (balanced) with augmentation
# - Paillier (quantized) + CKKS single + CKKS batched inference
# - Stratified K-fold CV and aggregated mean ± std reporting
# =============================================================================

# If running in Colab uncomment installs (only if needed)
# !pip install -q scikit-learn phe tenseal tqdm pandas
# PyG install depends on environment; assume torch & torch_geometric are installed.

import os
import time, random
import numpy as np
import torch
import torch.nn.functional as F
from copy import deepcopy
from tqdm import tqdm
import pandas as pd

from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, log_loss

from torch_geometric.datasets import TUDataset
from torch_geometric.loader import DataLoader
from torch_geometric.nn import GINConv, GATConv, global_mean_pool, global_max_pool, global_add_pool
from torch_geometric.utils import dropout_edge
from torch.nn import LayerNorm
import torch.nn as nn

import phe as paillier
import tenseal as ts

# ---------------- USER KNOBS ----------------
RND = 42
N_FOLDS = 5

# encoder
HIDDEN = 128
EMB_DIM = 256
HEADS = 4
DROP_EDGE_P = 0.0        # set >0.0 to enable DropEdge during training
DROPOUT = 0.4
ATTN_DROPOUT = 0.0       # Add attention dropout knob

# training
EPOCHS = 50
PATIENCE = 10
LR = 3e-4
WEIGHT_DECAY = 1e-4
LABEL_SMOOTH = 0.0       # 0 disables; try 0.05-0.1 for small datasets

# logistic regression augmentation
LR_AUG_COPIES = 2
LR_AUG_NOISE_STD = 1e-2

# PCA before HE (reduce dim to speed HE)
USE_PCA_BEFORE_HE = True
PCA_DIM = 32

# Paillier params
PAILLIER_SCALE = 5000
PAILLIER_KEY = 2048     # 1024 faster / less secure, 2048 better for demo

# CKKS params
CKKS_POLY = 8192
CKKS_CTX_SCALE_EXP = 40
CKKS_BATCH_SIZE = 8
CKKS_COEFFS = [60, 40, 40, 60]

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
# ---------------------------------------------

random.seed(RND); np.random.seed(RND); torch.manual_seed(RND)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(RND)
device = torch.device(DEVICE)
print("Device:", device)

# ---------------- Hybrid GAT+GIN++ ----------------
class HybridGINGATModel(nn.Module):
    def __init__(self, in_ch, hidden=HIDDEN, emb_dim=EMB_DIM, heads=HEADS, dropout=DROPOUT, attn_dropout=ATTN_DROPOUT):
        super().__init__()
        # first block: GAT from input -> hidden*heads, GIN -> hidden, concat -> hidden*heads + hidden
        self.gat1 = GATConv(in_ch, hidden, heads=heads, dropout=attn_dropout)
        self.gin1 = GINConv(nn.Sequential(nn.Linear(in_ch, hidden), nn.ReLU(), nn.Linear(hidden, hidden)))
        self.norm1 = LayerNorm(hidden * heads + hidden)
        self.res1 = nn.Linear(in_ch, hidden * heads + hidden)

        # second block: accepts hidden*heads + hidden
        self.gat2 = GATConv(hidden * heads + hidden, hidden, heads=heads, dropout=attn_dropout)
        self.gin2 = GINConv(nn.Sequential(nn.Linear(hidden * heads + hidden, hidden), nn.ReLU(), nn.Linear(hidden, hidden)))
        self.norm2 = LayerNorm(hidden * heads + hidden)
        self.res2 = nn.Linear(hidden * heads + hidden, hidden * heads + hidden)

        # gating and final projection to embedding dimension
        self.gate1 = nn.Parameter(torch.tensor(0.5))
        self.fc_out = nn.Linear(hidden * heads + hidden, emb_dim)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, edge_index, batch, dropedge_p=0.0, training=False):
        # x: node features, edge_index: edge list, batch: graph assignment vector
        if training and dropedge_p > 0:
            edge_index, _ = dropout_edge(edge_index, p=dropedge_p, force_undirected=True)

        # block 1
        gat1 = self.gat1(x, edge_index)                   # (num_nodes, hidden*heads)
        gin1 = self.gin1(x, edge_index)                   # (num_nodes, hidden)
        cat1 = torch.cat([gat1, gin1], dim=-1)            # (num_nodes, hidden*heads + hidden)
        res = self.res1(x)                                # project input -> same dim
        h1 = F.relu(self.norm1(cat1 + res))
        h1 = self.dropout(h1)

        # block 2
        gat2 = self.gat2(h1, edge_index)
        gin2 = self.gin2(h1, edge_index)
        cat2 = torch.cat([gat2, gin2], dim=-1)
        res2 = self.res2(h1)
        h2 = F.relu(self.norm2(cat2 + res2))
        h2 = self.dropout(h2)

        # Fusion and final output (node-level)
        g = torch.sigmoid(self.gate1)
        fused = g * h2 + (1.0 - g) * h1  # (num_nodes, hidden*heads + hidden)
        out_nodes = self.fc_out(fused)    # node-level embeddings (num_nodes, emb_dim)

        # pool to graph-level embedding
        graph_emb = global_mean_pool(out_nodes, batch)  # (n_graphs_in_batch, emb_dim)
        return graph_emb

    def extract_emb_batch(self, loader, device=device):
        """
        Extract graph-level embeddings (numpy) and labels for a DataLoader.
        """
        self.eval()
        Xs = []
        ys = []
        with torch.no_grad():
            for data in loader:
                data = data.to(device)
                emb = self(data.x.float(), data.edge_index, data.batch, dropedge_p=0.0, training=False)
                Xs.append(emb.cpu().numpy())
                ys.append(data.y.cpu().numpy())
        if len(Xs) == 0:
            return np.zeros((0, self.fc_out.out_features)), np.array([], dtype=int)
        X = np.vstack(Xs)
        y = np.concatenate(ys)
        return X, y

# ---------------- Training helper (early stopping) ----------------
def train_encoder(dataset, train_idx, val_idx, epochs=EPOCHS, patience=PATIENCE,
                  dropedge_p=DROP_EDGE_P, label_smooth=LABEL_SMOOTH):
    train_ds = dataset[torch.tensor(train_idx)]
    val_ds = dataset[torch.tensor(val_idx)]
    train_loader = DataLoader(train_ds, batch_size=32, shuffle=True)
    val_loader = DataLoader(val_ds, batch_size=32)

    model = HybridGINGATModel(dataset.num_features).to(device)
    head = torch.nn.Linear(EMB_DIM, dataset.num_classes).to(device)
    opt = torch.optim.AdamW(list(model.parameters()) + list(head.parameters()), lr=LR, weight_decay=WEIGHT_DECAY)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=max(1, epochs))
    crit = torch.nn.CrossEntropyLoss()
    # determine number of classes safely
    n_classes = int(torch.unique(torch.tensor([d.y for d in dataset])).numel())

    best_val = float("inf"); wait = 0; best_state = None

    for ep in range(1, epochs+1):
        model.train(); head.train()
        tloss = 0.0
        total_graphs = 0
        for data in train_loader:
            data = data.to(device)
            opt.zero_grad()
            emb = model(data.x.float(), data.edge_index, data.batch, dropedge_p=dropedge_p, training=True)  # (batch_graphs, emb)
            logits = head(emb)  # (batch_graphs, n_classes)
            if label_smooth > 0:
                # label smoothing implemented via smoothed one-hot
                y = torch.full((data.y.size(0), n_classes), fill_value=label_smooth / (n_classes - 1), device=device)
                y[torch.arange(data.y.size(0)), data.y.long()] = 1.0 - label_smooth
                loss = - (y * F.log_softmax(logits, dim=1)).sum(dim=1).mean()
            else:
                loss = crit(logits, data.y)
            loss.backward(); opt.step()
            tloss += float(loss.item()) * data.num_graphs
            total_graphs += data.num_graphs
        scheduler.step()
        if total_graphs > 0:
            tloss /= total_graphs
        else:
            tloss = 0.0

        # validation
        model.eval(); head.eval()
        vloss = 0.0
        total_val = 0
        with torch.no_grad():
            for data in val_loader:
                data = data.to(device)
                emb = model(data.x.float(), data.edge_index, data.batch, dropedge_p=0.0, training=False)
                out = head(emb)
                vloss += float(crit(out, data.y).item()) * data.num_graphs
                total_val += data.num_graphs
        vloss = vloss / max(1, total_val)

        # track best
        if vloss < best_val - 1e-6:
            best_val = vloss; wait = 0
            best_state = (deepcopy(model.state_dict()), deepcopy(head.state_dict()))
        else:
            wait += 1

        if ep % 10 == 0 or ep == 1:
            print(f"Epoch {ep:03d} train_loss: {tloss:.4f} val_loss: {vloss:.4f} wait: {wait}")

        if wait >= patience:
            print(f"Early stopping at epoch {ep} (wait={wait} >= patience={patience})")
            break

    if best_state is not None:
        model.load_state_dict(best_state[0]); head.load_state_dict(best_state[1])
    # return only the encoder model (for embedding extraction). Keep head saved inside best_state if needed.
    return model

# ---------------- Metrics ----------------
def softmax_rows(logits):
    l = np.array(logits, dtype=float)
    l = l - np.max(l, axis=1, keepdims=True)
    ex = np.exp(l)
    return ex / np.sum(ex, axis=1, keepdims=True)

def metrics_from_probs_multiclass(probs, y_true):
    preds = np.argmax(probs, axis=1)
    try:
        roc = roc_auc_score(y_true, probs, multi_class='ovr')
    except Exception:
        roc = float("nan")
    return {
        "accuracy": float(accuracy_score(y_true, preds)),
        "precision_macro": float(precision_score(y_true, preds, average='macro', zero_division=0)),
        "recall_macro": float(recall_score(y_true, preds, average='macro', zero_division=0)),
        "f1_macro": float(f1_score(y_true, preds, average='macro', zero_division=0)),
        "roc_auc_ovr": float(roc) if not np.isnan(roc) else float("nan"),
        "log_loss": float(log_loss(y_true, probs))
    }

def logits_to_probs_binary(logits):
    """
    Convert 1D array of logits (log-odds) to two-column probability array [[p0, p1], ...],
    where p1 = sigmoid(logit).
    """
    logits = np.asarray(logits).reshape(-1)
    p1 = 1.0 / (1.0 + np.exp(-logits))
    p0 = 1.0 - p1
    probs = np.vstack([p0, p1]).T
    return probs

def metrics_from_probs(probs, y_true):
    """
    Wrapper that handles binary (n,2) probs and multiclass.
    """
    probs = np.asarray(probs)
    if probs.ndim == 1 or (probs.ndim == 2 and probs.shape[1] == 1):
        # treat as logits or single-prob-of-class1
        if probs.ndim == 2:
            probs = probs.reshape(-1)
        probs2 = logits_to_probs_binary(probs)
        return metrics_from_probs_multiclass(probs2, y_true)
    elif probs.ndim == 2 and probs.shape[1] == 2:
        return metrics_from_probs_multiclass(probs, y_true)
    else:
        # general multiclass case
        return metrics_from_probs_multiclass(probs, y_true)

# ---------------- Paillier inference (binary, per-sample) ----------------
def paillier_inference(X, w, b, scale=PAILLIER_SCALE, key_size=PAILLIER_KEY):
    # X: (n_samples, d), w: (d,), b: scalar-array-like
    pk, sk = paillier.generate_paillier_keypair(n_length=key_size)
    w_int = [int(round(float(wi) * scale)) for wi in w.tolist()]
    # bias scaled by scale^2 (because dot products are scale^2)
    b_int = int(round(float(b[0]) * (scale ** 2))) if hasattr(b, "__len__") else int(round(float(b) * (scale ** 2)))
    logits = []
    t0 = time.time()
    for x in X:
        # quantize x to integers
        x_int = [int(round(float(xj) * scale)) for xj in x.tolist()]
        # encrypt features
        enc_x = [pk.encrypt(int(v)) for v in x_int]
        acc = pk.encrypt(int(b_int))
        for enc_feat, wi in zip(enc_x, w_int):
            acc = acc + (enc_feat * int(wi))
        dec_int = sk.decrypt(acc)
        score = float(dec_int) / (scale ** 2)
        logits.append(score)
    wall = time.time() - t0
    return np.array(logits), wall

# ---------------- CKKS single inference ----------------
def ckks_single_inference(X, w, b, poly_mod=CKKS_POLY, ctx_scale_exp=CKKS_CTX_SCALE_EXP, coeff_mod_bit_sizes=CKKS_COEFFS):
    ctx = ts.context(ts.SCHEME_TYPE.CKKS, poly_modulus_degree=poly_mod, coeff_mod_bit_sizes=coeff_mod_bit_sizes)
    ctx.global_scale = 2 ** ctx_scale_exp
    ctx.generate_galois_keys()
    # plaintext weight vector packing onto CKKS (we will dot-encrypt x with plaintext w if supported)
    # For safety we pack both as vectors and do encrypted dot
    logits = []
    t0 = time.time()
    for x in X:
        enc_x = ts.ckks_vector(ctx, x.tolist())
        enc_w = ts.ckks_vector(ctx, w.tolist())
        enc_dot = enc_x.dot(enc_w)
        dec_dot = enc_dot.decrypt()
        # dec_dot sometimes returns list or scalar-like
        val = float(dec_dot[0]) if hasattr(dec_dot, "__len__") else float(dec_dot)
        val = val + float(b[0]) if hasattr(b, "__len__") else val + float(b)
        # compute exact sigmoid in plaintext then convert to logit for consistency
        ppos = 1.0 / (1.0 + np.exp(-val))
        eps = 1e-12
        logit = np.log(max(eps, ppos) / max(eps, 1-ppos))
        logits.append(logit)
    wall = time.time() - t0
    return np.array(logits), wall

# ---------------- CKKS batched inference ----------------
def ckks_batched_inference(X, w, b, batch_size=CKKS_BATCH_SIZE, poly_mod=CKKS_POLY, ctx_scale_exp=CKKS_CTX_SCALE_EXP, coeff_mod_bit_sizes=CKKS_COEFFS):
    ctx = ts.context(ts.SCHEME_TYPE.CKKS, poly_modulus_degree=poly_mod, coeff_mod_bit_sizes=coeff_mod_bit_sizes)
    ctx.global_scale = 2 ** ctx_scale_exp
    ctx.generate_galois_keys()
    n = X.shape[0]
    logits = []
    t0 = time.time()
    for start in range(0, n, batch_size):
        end = min(start + batch_size, n)
        batch = X[start:end]   # shape (bs, d)
        bs = batch.shape[0]
        # compute weighted sum across features by encrypting columns (pack per-sample values)
        weighted_sum = None
        for j in range(batch.shape[1]):
            col_vals = batch[:, j].astype(float).tolist()  # length bs
            enc_col = ts.ckks_vector(ctx, col_vals)
            term = enc_col * float(w[j])
            weighted_sum = term if weighted_sum is None else (weighted_sum + term)
        bias_vec = ts.ckks_vector(ctx, [float(b[0])] * bs) if hasattr(b, "__len__") else ts.ckks_vector(ctx, [float(b)] * bs)
        enc_logits = weighted_sum + bias_vec
        dec = enc_logits.decrypt()   # list length bs
        for v in dec:
            val = float(v) if np.isfinite(v) else 0.0
            ppos = 1.0 / (1.0 + np.exp(-val))
            eps = 1e-12
            logit = np.log(max(eps, ppos) / max(eps, 1-ppos))
            logits.append(logit)
    wall = time.time() - t0
    return np.array(logits), wall

# ---------------- main CV loop ----------------
def run_full_pipeline():
    print("Loading MUTAG dataset...")
    dataset = TUDataset(root="/tmp/MUTAG_improved", name="MUTAG")
    n = len(dataset)
    labels = np.array([int(d.y.item()) for d in dataset])
    print("Graphs:", n, "Classes:", np.unique(labels), "Features:", dataset.num_features)

    skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=RND)

    fold_results = []
    fold = 0
    for trainval_idx, test_idx in skf.split(np.arange(n), labels):
        fold += 1
        print(f"\n=== Fold {fold}/{N_FOLDS} ===")
        train_idx, val_idx = train_test_split(trainval_idx, test_size=0.125, random_state=RND, stratify=labels[trainval_idx])
        print("train/val/test sizes:", len(train_idx), len(val_idx), len(test_idx))

        # 1) train encoder
        encoder = train_encoder(dataset, train_idx, val_idx, epochs=EPOCHS, patience=PATIENCE,
                                dropedge_p=DROP_EDGE_P, label_smooth=LABEL_SMOOTH)

        # 2) extract embeddings
        train_loader = DataLoader(dataset[torch.tensor(train_idx)], batch_size=32)
        val_loader = DataLoader(dataset[torch.tensor(val_idx)], batch_size=32)
        test_loader = DataLoader(dataset[torch.tensor(test_idx)], batch_size=32)

        X_train, y_train = encoder.extract_emb_batch(train_loader)
        X_val, y_val = encoder.extract_emb_batch(val_loader)
        X_test, y_test = encoder.extract_emb_batch(test_loader)
        print("Emb shapes (train/val/test):", X_train.shape, X_val.shape, X_test.shape)

        # 3) scale embeddings and optionally PCA before HE
        scaler = StandardScaler()
        scaler.fit(np.vstack([X_train, X_val]))
        X_train_s = scaler.transform(X_train)
        X_val_s   = scaler.transform(X_val)
        X_test_s  = scaler.transform(X_test)

        if USE_PCA_BEFORE_HE:
            pca = PCA(n_components=min(PCA_DIM, X_train_s.shape[1]), random_state=RND)
            pca.fit(np.vstack([X_train_s, X_val_s]))
            X_train_he = pca.transform(X_train_s)
            X_val_he   = pca.transform(X_val_s)
            X_test_he  = pca.transform(X_test_s)
        else:
            X_train_he, X_val_he, X_test_he = X_train_s, X_val_s, X_test_s

        # LR augmentation (augment training set with small noise copies)
        def augment(X, y, copies=LR_AUG_COPIES, noise_std=LR_AUG_NOISE_STD):
            Xs, ys = [X], [y]
            for _ in range(copies):
                Xs.append(X + np.random.normal(scale=noise_std, size=X.shape))
                ys.append(y.copy())
            return np.vstack(Xs), np.concatenate(ys)

        X_aug, y_aug = augment(np.vstack([X_train_s, X_val_s]), np.concatenate([y_train, y_val]))

        # 4) train logistic regression plaintext baseline
        clf = LogisticRegression(max_iter=5000, solver="lbfgs", class_weight="balanced")
        clf.fit(X_aug, y_aug)

        probs_plain = clf.predict_proba(X_test_s)
        metrics_plain = metrics_from_probs(probs_plain, y_test)

        # prepare weights for HE (train a small LR in PCA/HE space to get W/b)
        clf_he = LogisticRegression(max_iter=5000, solver="lbfgs", class_weight="balanced")
        clf_he.fit(np.vstack([X_train_he, X_val_he]), np.concatenate([y_train, y_val]))
        W_he = clf_he.coef_.astype(float)
        b_he = clf_he.intercept_.astype(float)
        if W_he.ndim == 1:
            W_he = W_he.reshape(1, -1)

        # For binary convert to vector
        if W_he.shape[0] == 1:
            w_vec = W_he[0]
        else:
            # OvR multiclass could be supported; MUTAG is binary so this is fine
            w_vec = W_he[0]

        # 5) Paillier inference (per-sample)
        paillier_logits, paillier_time = paillier_inference(X_test_he, w_vec, b_he, scale=PAILLIER_SCALE, key_size=PAILLIER_KEY)
        probs_paillier = logits_to_probs_binary(paillier_logits)
        metrics_paillier = metrics_from_probs(probs_paillier, y_test)

        # 6) CKKS single-sample
        ckks_single_logits, ckks_single_time = ckks_single_inference(X_test_he, w_vec, b_he,
                                                                      poly_mod=CKKS_POLY,
                                                                      ctx_scale_exp=CKKS_CTX_SCALE_EXP,
                                                                      coeff_mod_bit_sizes=CKKS_COEFFS)
        probs_ckks_single = logits_to_probs_binary(ckks_single_logits)
        metrics_ckks_single = metrics_from_probs(probs_ckks_single, y_test)

        # 7) CKKS batched
        ckks_batch_logits, ckks_batch_time = ckks_batched_inference(X_test_he, w_vec, b_he,
                                                                    batch_size=CKKS_BATCH_SIZE,
                                                                    poly_mod=CKKS_POLY,
                                                                    ctx_scale_exp=CKKS_CTX_SCALE_EXP,
                                                                    coeff_mod_bit_sizes=CKKS_COEFFS)
        probs_ckks_batch = logits_to_probs_binary(ckks_batch_logits)
        metrics_ckks_batch = metrics_from_probs(probs_ckks_batch, y_test)

        print("Fold results (Plain / Paillier / CKKS single / CKKS batched):")
        print("Plain:", metrics_plain)
        print("Paillier:", metrics_paillier)
        print("CKKS single:", metrics_ckks_single)
        print("CKKS batch:", metrics_ckks_batch)
        print("Runtimes (s): paillier=%.2f, ckks_single=%.2f, ckks_batch=%.2f" % (paillier_time, ckks_single_time, ckks_batch_time))

        fold_results.append({
            "plain": metrics_plain, "paillier": metrics_paillier,
            "ckks_single": metrics_ckks_single, "ckks_batch": metrics_ckks_batch,
            "time_paillier": paillier_time, "time_ckks_single": ckks_single_time, "time_ckks_batch": ckks_batch_time
        })

    # ---------- aggregate ----------
    def agg(key):
        arr = [fr[key] for fr in fold_results]
        df = pd.DataFrame(arr)
        mean = df.mean().to_dict(); std = df.std().to_dict()
        return {k: f"{mean[k]:.4f} ± {std[k]:.4f}" for k in mean.keys()}

    print("\n=== Aggregated (mean ± std) ===")
    print("Plaintext:", agg("plain"))
    print("Paillier :", agg("paillier"))
    print("CKKS sing:", agg("ckks_single"))
    print("CKKS bat :", agg("ckks_batch"))

    # runtimes
    times = pd.DataFrame([{"paillier":fr["time_paillier"], "ckks_single":fr["time_ckks_single"], "ckks_batch":fr["time_ckks_batch"]} for fr in fold_results])
    print("\nAverage runtimes (s):", times.mean().to_dict())

    print("\nNotes:")
    print("- Ensemble_size, EPOCHS, and model dims control accuracy vs runtime.")
    print("- Reduce ENSEMBLE_SIZE or EPOCHS to speed up.")

# -------------- run --------------
if __name__ == "__main__":
    # safety check: avoid torch.py shadowing problems
    if os.path.exists("torch.py") or os.path.exists("torch.pyc"):
        raise RuntimeError("Please rename/remove any local file named 'torch.py' or 'torch.pyc' that shadows the real PyTorch package.")
    run_full_pipeline()


Device: cuda
Loading MUTAG dataset...
Graphs: 188 Classes: [0 1] Features: 7

=== Fold 1/5 ===
train/val/test sizes: 131 19 38
Epoch 001 train_loss: 0.6387 val_loss: 0.7093 wait: 0
Epoch 010 train_loss: 0.6318 val_loss: 0.5650 wait: 3
Epoch 020 train_loss: 0.5598 val_loss: 0.5618 wait: 3
Epoch 030 train_loss: 0.5173 val_loss: 0.5015 wait: 1
Epoch 040 train_loss: 0.5262 val_loss: 0.4944 wait: 0
Epoch 050 train_loss: 0.5140 val_loss: 0.4943 wait: 8
Emb shapes (train/val/test): (131, 256) (19, 256) (38, 256)
Fold results (Plain / Paillier / CKKS single / CKKS batched):
Plain: {'accuracy': 0.9210526315789473, 'precision_macro': 0.9198717948717949, 'recall_macro': 0.9030769230769231, 'f1_macro': 0.9105882352941177, 'roc_auc_ovr': nan, 'log_loss': 0.2488345235188758}
Paillier: {'accuracy': 0.9210526315789473, 'precision_macro': 0.9198717948717949, 'recall_macro': 0.9030769230769231, 'f1_macro': 0.9105882352941177, 'roc_auc_ovr': nan, 'log_loss': 0.23642587537688806}
CKKS single: {'accuracy':

In [1]:
# pip installs (uncomment in Colab if needed)
!pip install -q scikit-learn phe tenseal tqdm
!pip install torch_geometric

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.7/53.7 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.8/4.8 MB 82.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.1/63.1 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 35.4 MB/s eta 0:00:00
